In [1]:
import os, json, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
import torchvision.models as tv_models

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"PyTorch: {torch.__version__}")

# Paths — adjust to match your Drive layout
DATA_DIR   = Path('/content/drive/MyDrive/Projects_Data/Speech_to_Emotions/emotion_data/processed_v2')
OUTPUT_DIR = Path('/content/drive/MyDrive/Projects_Data/Speech_to_Emotions/emotion_data/model_v2')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Must match preprocessing constants exactly
N_MELS      = 64
N_FRAMES    = 125
NUM_CLASSES = 6
BATCH_SIZE  = 32
EPOCHS      = 100

Device : cuda
PyTorch: 2.10.0+cu128


In [5]:
X_clean = np.load(DATA_DIR / 'X_clean.npy')   # (N, 1, 64, 125)
y_clean = np.load(DATA_DIR / 'y_clean.npy')   # (N,)

aug_exists = (DATA_DIR / 'X_aug.npy').exists()
if aug_exists:
    X_aug = np.load(DATA_DIR / 'X_aug.npy')
    y_aug = np.load(DATA_DIR / 'y_aug.npy')
    print(f"Augmented data found: {X_aug.shape}")

with open(DATA_DIR / 'label_map.json') as f:
    lm           = json.load(f)
    LABEL_TO_INT = lm['label_to_int']
    INT_TO_LABEL = {int(k): v for k, v in lm['int_to_label'].items()}

print(f"Clean data  — X: {X_clean.shape}  y: {y_clean.shape}")
assert not np.isnan(X_clean).any() and not np.isinf(X_clean).any(), "Data has NaN/Inf!"

# Index-based split: keeps augmented data correctly linked to training indices only
indices    = np.arange(len(X_clean))
train_idx, temp_idx = train_test_split(
    indices,  test_size=0.30, stratify=y_clean, random_state=SEED)
val_idx, test_idx   = train_test_split(
    temp_idx, test_size=0.50, stratify=y_clean[temp_idx], random_state=SEED)

X_tr,   y_tr   = X_clean[train_idx], y_clean[train_idx]
X_val,  y_val  = X_clean[val_idx],   y_clean[val_idx]
X_test, y_test = X_clean[test_idx],  y_clean[test_idx]

# Augmented copies go into TRAINING ONLY — never val, never test
if aug_exists:
    X_tr = np.concatenate([X_tr, X_aug[train_idx]], axis=0)
    y_tr = np.concatenate([y_tr, y_clean[train_idx]], axis=0)

print(f"\nTrain : {len(y_tr):>5}  (incl. aug)")
print(f"Val   : {len(y_val):>5}")
print(f"Test  : {len(y_test):>5}")
print(f"\nTrain class distribution:")
for i in range(NUM_CLASSES):
    bar = '█' * ((y_tr == i).sum() // 30)
    print(f"  {INT_TO_LABEL[i]:<10}  {(y_tr==i).sum():>5}  {bar}")

Augmented data found: (9938, 1, 64, 126)
Clean data  — X: (9938, 1, 64, 126)  y: (9938,)

Train : 13912  (incl. aug)
Val   :  1491
Test  :  1491

Train class distribution:
  angry        2316  █████████████████████████████████████████████████████████████████████████████
  disgust      2318  █████████████████████████████████████████████████████████████████████████████
  fear         2316  █████████████████████████████████████████████████████████████████████████████
  happy        2316  █████████████████████████████████████████████████████████████████████████████
  neutral      2328  █████████████████████████████████████████████████████████████████████████████
  sad          2318  █████████████████████████████████████████████████████████████████████████████


In [6]:
class SpectrogramDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):         return len(self.y)
    def __getitem__(self, i):  return self.X[i], self.y[i]

def make_loader(X, y, batch_size, weighted=False, shuffle=False):
    ds = SpectrogramDataset(X, y)
    sampler = None
    if weighted:
        counts  = np.bincount(y, minlength=NUM_CLASSES).astype(np.float32)
        w       = 1.0 / (counts + 1e-8)
        sampler = WeightedRandomSampler(
            torch.tensor(w[y], dtype=torch.float32),
            num_samples=len(y), replacement=True)
        shuffle = False   # mutually exclusive with sampler
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      sampler=sampler, num_workers=2,
                      pin_memory=(DEVICE.type == 'cuda'))

train_loader = make_loader(X_tr,   y_tr,   BATCH_SIZE, weighted=True)
val_loader   = make_loader(X_val,  y_val,  BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

# Shape sanity check
xb, yb = next(iter(train_loader))
print(f"Batch X : {tuple(xb.shape)}   ← (B, C, mel_bands, time_frames)")
print(f"Batch y : {tuple(yb.shape)}")
print(f"Loaders → train: {len(train_loader)} batches | val: {len(val_loader)} | test: {len(test_loader)}")

Batch X : (32, 1, 64, 126)   ← (B, C, mel_bands, time_frames)
Batch y : (32,)
Loaders → train: 435 batches | val: 47 | test: 47


In [7]:
# ── Focal Loss ───────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """
    Down-weights easy correct predictions, focuses learning on hard wrong ones.
    gamma=2.0 is the standard value from Lin et al. (2017).
    label_smoothing prevents the model from becoming over-confident.
    """
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.1):
        super().__init__()
        self.gamma = gamma; self.weight = weight; self.ls = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight,
                             label_smoothing=self.ls, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

# Build class-weighted focal loss (fit on training distribution only)
counts       = np.bincount(y_tr, minlength=NUM_CLASSES).astype(np.float32)
class_w      = (1.0 / counts) / (1.0 / counts).sum() * NUM_CLASSES
cw_tensor    = torch.tensor(class_w, dtype=torch.float32).to(DEVICE)
criterion    = FocalLoss(gamma=2.0, weight=cw_tensor, label_smoothing=0.1)
print(f"Criterion: FocalLoss(gamma=2, smooth=0.1)")
print(f"Class weights: { {INT_TO_LABEL[i]: f'{class_w[i]:.3f}' for i in range(NUM_CLASSES)} }")

# ── Unweighted Accuracy (UA) ─────────────────────────────────────────────────
# UA = mean per-class recall — fairer metric than WA when classes are imbalanced
def ua_score(preds, labels):
    per_class = []
    for c in range(NUM_CLASSES):
        mask = labels == c
        if mask.sum() == 0: continue
        per_class.append((preds[mask] == c).mean())
    return float(np.mean(per_class))

# ── One training epoch ────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    tot_loss = correct = n = 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        with torch.no_grad():
            preds    = model(X).argmax(-1)
            correct += (preds == y).sum().item()
            tot_loss += loss.item() * len(y)
            n        += len(y)
    return tot_loss / n, correct / n

# ── Evaluation ────────────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    tot_loss = correct = n = 0
    all_preds, all_labels = [], []
    for X, y in loader:
        X, y   = X.to(device), y.to(device)
        logits = model(X)
        loss   = criterion(logits, y)
        preds  = logits.argmax(-1)
        correct  += (preds == y).sum().item()
        tot_loss += loss.item() * len(y)
        n        += len(y)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y.cpu().numpy())
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    wa = correct / n
    ua = ua_score(all_preds, all_labels)
    return tot_loss / n, wa, ua, all_preds, all_labels

# ── Universal training loop (shared by ALL models) ────────────────────────────
def train_model(model, name, cfg):
    model = model.to(DEVICE)
    opt   = AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['wd'])

    # Linear warmup → cosine decay
    warmup = LinearLR(opt, start_factor=0.1, end_factor=1.0, total_iters=cfg['warmup'])
    cosine = CosineAnnealingLR(opt, T_max=cfg['epochs'] - cfg['warmup'], eta_min=1e-6)
    sched  = SequentialLR(opt, [warmup, cosine], milestones=[cfg['warmup']])

    best_ua, best_wts = 0.0, copy.deepcopy(model.state_dict())
    patience = 0
    history  = {'tl': [], 'ta': [], 'vl': [], 'va': [], 'vu': []}
    t0       = time.time()

    print(f"\n{'═'*68}")
    print(f"  {name}   |   lr={cfg['lr']}  wd={cfg['wd']}  patience={cfg['patience']}")
    print(f"{'─'*68}")
    print(f"{'Ep':>4} | {'TrLoss':>8} | {'TrWA':>6} | {'VaLoss':>8} | {'VaWA':>6} | {'VaUA':>6}")
    print(f"{'─'*68}")

    for ep in range(1, cfg['epochs'] + 1):
        tl, ta          = train_one_epoch(model, train_loader, opt, criterion, DEVICE)
        vl, va, vu, *_  = evaluate(model, val_loader, criterion, DEVICE)
        sched.step()

        for k, v in zip(['tl','ta','vl','va','vu'], [tl,ta,vl,va,vu]):
            history[k].append(v)

        marker = ''
        if vu > best_ua:
            best_ua, best_wts = vu, copy.deepcopy(model.state_dict())
            torch.save(best_wts, OUTPUT_DIR / f'{name}_best.pt')
            patience = 0
            marker   = '  ✓'
        else:
            patience += 1

        print(f"{ep:>4} | {tl:>8.4f} | {ta:>5.2%} | {vl:>8.4f} | {va:>5.2%} | {vu:>5.2%}{marker}")

        if patience >= cfg['patience']:
            print(f"  ↳ Early stop at epoch {ep}")
            break

    model.load_state_dict(best_wts)
    print(f"\n  Done in {(time.time()-t0)/60:.1f} min  |  Best Val UA: {best_ua:.2%}")
    return model, best_ua, history

# Store test results for the final comparison
all_results = {}
print("\nAll training utilities defined and ready.")

Criterion: FocalLoss(gamma=2, smooth=0.1)
Class weights: {'angry': '1.001', 'disgust': '1.000', 'fear': '1.001', 'happy': '1.001', 'neutral': '0.996', 'sad': '1.000'}

All training utilities defined and ready.


#### Model1: ResNet-18
##### Standard ResNet-18 backbone adapted for 1-channel mel-spectrograms.

- WHY ResNet-18 for SER?
    - ResNet's residual connections let it train very deep without degradation.
    - In SER literature it consistently outperforms shallow CNNs on 2D
    - mel-spectrograms because it can learn both low-level texture (voice quality)
    - and high-level shape (prosodic contour) via its staged feature hierarchy.
    - The only modification needed: change conv1 from 3→1 channel (mel-specs
    - are single-channel, unlike RGB images).

In [8]:
class ResNet18SER(nn.Module):
    def __init__(self, num_classes=6, dropout=0.4):
        super().__init__()
        backbone = tv_models.resnet18(weights=None)   # train from scratch

        # Original: Conv2d(3, 64, 7, stride=2, padding=3)
        # Ours:     Conv2d(1, 64, 7, stride=2, padding=3)
        backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7,
                                   stride=2, padding=3, bias=False)

        # Remove ResNet's own classifier — we add a custom head
        self.backbone = nn.Sequential(
            backbone.conv1, backbone.bn1, backbone.relu, backbone.maxpool,
            backbone.layer1,   # 64 ch — low-level edges, textures
            backbone.layer2,   # 128 ch — formant shapes
            backbone.layer3,   # 256 ch — prosodic patterns
            backbone.layer4,   # 512 ch — full emotion features
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))   # global spatial average
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.GELU(),
            nn.Dropout(dropout * 0.6),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):          # x: (B, 1, 64, 125)
        x = self.backbone(x)       # (B, 512, H', W')
        x = self.pool(x)           # (B, 512, 1, 1)
        return self.head(x)        # (B, 6)

# Dry run
_m = ResNet18SER(NUM_CLASSES)
_o = _m(torch.randn(4, 1, N_MELS, N_FRAMES))
print(f"ResNet-18 SER")
print(f"  Params : {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"  Output : {tuple(_o.shape)}")
del _m, _o

ResNet-18 SER
  Params : 11,303,110
  Output : (4, 6)


In [9]:
cfg_r18 = {
    'lr':       1e-3,   # CNN from scratch can handle a higher LR than transformers
    'wd':       1e-4,
    'epochs':   EPOCHS,
    'warmup':   5,
    'patience': 20,
}

model_r18, _, hist_r18 = train_model(
    ResNet18SER(NUM_CLASSES, dropout=0.4), 'ResNet18', cfg_r18)

_, wa_r18, ua_r18, preds_r18, labels_r18 = evaluate(
    model_r18, test_loader, criterion, DEVICE)

all_results['ResNet18'] = {
    'wa': wa_r18, 'ua': ua_r18,
    'preds': preds_r18, 'labels': labels_r18, 'history': hist_r18}
print(f"\n{'─'*40}")
print(f"ResNet-18  |  Test WA: {wa_r18:.2%}  Test UA: {ua_r18:.2%}")
print(f"{'─'*40}")


════════════════════════════════════════════════════════════════════
  ResNet18   |   lr=0.001  wd=0.0001  patience=20
────────────────────────────────────────────────────────────────────
  Ep |   TrLoss |   TrWA |   VaLoss |   VaWA |   VaUA
────────────────────────────────────────────────────────────────────
   1 |   0.9597 | 47.26% |   0.8649 | 47.42% | 47.41%  ✓
   2 |   0.8252 | 61.98% |   0.8341 | 52.25% | 52.23%  ✓
   3 |   0.7590 | 66.89% |   0.7723 | 54.19% | 54.19%  ✓
   4 |   0.6992 | 70.37% |   0.9472 | 45.27% | 45.25%
   5 |   0.6577 | 72.85% |   0.7875 | 55.40% | 55.42%  ✓
   6 |   0.5963 | 77.68% |   0.7530 | 58.48% | 58.48%  ✓
   7 |   0.4907 | 83.85% |   0.8370 | 57.68% | 57.69%
   8 |   0.4031 | 89.05% |   0.8078 | 58.95% | 58.94%  ✓
   9 |   0.3387 | 92.29% |   0.9803 | 56.94% | 56.94%
  10 |   0.2887 | 94.76% |   0.9225 | 57.28% | 57.26%
  11 |   0.2458 | 96.33% |   0.8943 | 58.69% | 58.69%
  12 |   0.2182 | 97.35% |   0.9912 | 59.02% | 59.02%  ✓
  13 |   0.1851 | 9

#### Model 2: CNN-BiLSTM-Attemtion

##### The dominant SER architecture in literature (2018–2023).

- WHY this design?
    - CNN handles the 2D structure of the mel-spec — local time-frequency patterns like formant transitions or pitch bends.
    - BiLSTM sees the resulting frame sequence and captures long-range temporal dependencies (e.g., rising intonation across 1-2 seconds).
    - Attention pool is the key improvement over plain mean/max pooling:
      - it learns to upweight the 5-10 most emotionally charged frames
      (the peak of an angry shout, the tremor in a fearful phrase) and
      downweight silent or neutral-sounding regions.

In [10]:
class CNNBiLSTMAttention(nn.Module):

    def __init__(self, num_classes=6, cnn_out=256,
                 lstm_hidden=128, lstm_layers=2, dropout=0.4):
        super().__init__()

        # Frequency-only pooling: time axis stays at 125 throughout
        def conv_block(cin, cout, freq_pool):
            return nn.Sequential(
                nn.Conv2d(cin, cout, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(cout), nn.GELU(), nn.Dropout2d(0.1),
                nn.MaxPool2d(kernel_size=(freq_pool, 1))  # ↓ freq only
            )

        self.cnn = nn.Sequential(
            conv_block(1,       32,      2),   # (B, 32,  32, 125)
            conv_block(32,      64,      2),   # (B, 64,  16, 125)
            conv_block(64,      128,     2),   # (B, 128,  8, 125)
            conv_block(128,  cnn_out,    8),   # (B, 256,  1, 125)
        )

        self.bilstm = nn.LSTM(
            input_size=cnn_out, hidden_size=lstm_hidden, num_layers=lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0)
        lstm_out = lstm_hidden * 2   # bidirectional

        # Attention: one scalar score per time frame
        self.attn_fc = nn.Linear(lstm_out, 1)
        self.drop    = nn.Dropout(dropout)
        self.head    = nn.Sequential(
            nn.Linear(lstm_out, lstm_out // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_out // 2, num_classes)
        )

    def forward(self, x):           # x: (B, 1, 64, 125)
        x = self.cnn(x)             # (B, 256, 1, 125)
        x = x.squeeze(2)            # (B, 256, 125)
        x = x.permute(0, 2, 1)     # (B, 125, 256) — 125-step sequence

        x, _ = self.bilstm(x)      # (B, 125, 256)
        x    = self.drop(x)

        # Soft attention over 125 time steps
        w = torch.softmax(self.attn_fc(x), dim=1)  # (B, 125, 1)
        x = (w * x).sum(dim=1)                     # (B, 256)
        return self.head(x)                         # (B, 6)

_m = CNNBiLSTMAttention(NUM_CLASSES)
_o = _m(torch.randn(4, 1, N_MELS, N_FRAMES))
print(f"CNN-BiLSTM-Attention")
print(f"  Params : {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"  Output : {tuple(_o.shape)}")
del _m, _o

CNN-BiLSTM-Attention
  Params : 1,212,775
  Output : (4, 6)


In [11]:
cfg_clba = {
    'lr': 3e-4, 'wd': 3e-4, 'epochs': EPOCHS, 'warmup': 8, 'patience': 20}

model_clba, _, hist_clba = train_model(
    CNNBiLSTMAttention(NUM_CLASSES, dropout=0.4), 'CNN_BiLSTM_Attn', cfg_clba)

_, wa_clba, ua_clba, preds_clba, labels_clba = evaluate(
    model_clba, test_loader, criterion, DEVICE)

all_results['CNN_BiLSTM_Attn'] = {
    'wa': wa_clba, 'ua': ua_clba,
    'preds': preds_clba, 'labels': labels_clba, 'history': hist_clba}
print(f"\nCNN-BiLSTM-Attn  |  Test WA: {wa_clba:.2%}  Test UA: {ua_clba:.2%}")


════════════════════════════════════════════════════════════════════
  CNN_BiLSTM_Attn   |   lr=0.0003  wd=0.0003  patience=20
────────────────────────────────────────────────────────────────────
  Ep |   TrLoss |   TrWA |   VaLoss |   VaWA |   VaUA
────────────────────────────────────────────────────────────────────
   1 |   1.2202 | 21.54% |   1.1228 | 30.18% | 30.24%  ✓
   2 |   1.1156 | 31.49% |   1.0114 | 36.75% | 36.79%  ✓
   3 |   1.0318 | 36.96% |   0.9454 | 42.99% | 42.98%  ✓
   4 |   0.9866 | 40.48% |   0.8665 | 47.48% | 47.48%  ✓
   5 |   0.9372 | 43.49% |   0.8418 | 48.29% | 48.27%  ✓
   6 |   0.8990 | 47.15% |   0.8180 | 50.57% | 50.57%  ✓
   7 |   0.8667 | 48.58% |   0.7878 | 52.05% | 52.04%  ✓
   8 |   0.8513 | 49.90% |   0.7775 | 52.65% | 52.63%  ✓
   9 |   0.8372 | 50.57% |   0.7274 | 55.60% | 55.59%  ✓
  10 |   0.7942 | 52.90% |   0.7628 | 56.41% | 56.40%  ✓
  11 |   0.7687 | 55.00% |   0.7159 | 56.87% | 56.86%  ✓
  12 |   0.7526 | 55.98% |   0.7575 | 55.94% | 55.94%

#### Model 3: ARCNN

##### Attention-based Convolutional Recurrent Neural Network. 
###### `Reference: Chen et al., Interspeech 2018.`

- WHY different from CNN-BiLSTM-Attention?
    - In CNN-BiLSTM-Attn, attention is over TIME (which frames matter most).
    - In ACRNN, there are TWO levels of attention:
      1. Frequency attention (before the RNN): at each time step, the
         model learns which of the remaining mel bands are most relevant.
         This removes irrelevant frequency noise before temporal modeling.
         Result: the BiGRU sees a cleaner, more focused input.

      2. Time attention (after the RNN): selects the most salient frames
         from the BiGRU's output — same as in CNN-BiLSTM-Attn.

    - The critical difference: frequency noise is suppressed PER TIME STEP,
    not globally. 
    - A band can be important at one moment and irrelevant at
    another (e.g., high-freq breathiness only at sentence onset).

In [12]:
class ACRNN(nn.Module):

    def __init__(self, num_classes=6, dropout=0.4):
        super().__init__()

        # CNN: frequency-only pooling → time stays at 125
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d((2, 1)),   # (B,32,32,125)
            nn.Conv2d(32, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d((2, 1)),   # (B,64,16,125)
            nn.Conv2d(64, 128, 3, padding=1, bias=False),
            nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d((2, 1)),  # (B,128,8,125)
        )
        # Output: (B, 128 channels, 8 freq bins, 125 time steps)

        # Frequency attention: for each time step, score each of the 8 freq positions
        # Linear(128→1) applied to the channel vector at every (time, freq) position
        self.freq_score = nn.Linear(128, 1)

        # BiGRU on the freq-attended sequence
        self.bigru = nn.GRU(
            input_size=128, hidden_size=128,
            num_layers=2, batch_first=True,
            bidirectional=True, dropout=dropout)
        gru_out = 256   # 128 × 2

        # Time attention: two-layer MLP for richer scoring
        self.time_attn = nn.Sequential(
            nn.Linear(gru_out, 64), nn.Tanh(),
            nn.Linear(64, 1)
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(gru_out, 128), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):                          # (B, 1, 64, 125)
        x = self.cnn(x)                            # (B, 128, 8, 125)
        B, C, F, T = x.shape

        # Permute → (B, T, F, C) so we can apply attention over F per time step
        x = x.permute(0, 3, 2, 1)                 # (B, 125, 8, 128)

        # Score each freq position: Linear(128→1) → softmax over F=8
        freq_w = torch.softmax(
            self.freq_score(x),                    # (B, 125, 8, 1)
            dim=2                                  # softmax over 8 freq bins
        )
        x = (x * freq_w).sum(dim=2)               # (B, 125, 128)  ← freq collapsed

        # BiGRU over 125 frames
        x, _ = self.bigru(x)                       # (B, 125, 256)
        x    = self.drop(x)

        # Time attention
        time_w = torch.softmax(self.time_attn(x), dim=1)  # (B, 125, 1)
        x      = (time_w * x).sum(dim=1)                  # (B, 256)
        return self.head(x)

_m = ACRNN(NUM_CLASSES)
_o = _m(torch.randn(4, 1, N_MELS, N_FRAMES))
print(f"ACRNN")
print(f"  Params : {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"  Output : {tuple(_o.shape)}")
del _m, _o

ACRNN
  Params : 637,800
  Output : (4, 6)


In [13]:
cfg_acrnn = {
    'lr': 3e-4, 'wd': 3e-4, 'epochs': EPOCHS, 'warmup': 8, 'patience': 20}

model_acrnn, _, hist_acrnn = train_model(
    ACRNN(NUM_CLASSES, dropout=0.4), 'ACRNN', cfg_acrnn)

_, wa_acrnn, ua_acrnn, preds_acrnn, labels_acrnn = evaluate(
    model_acrnn, test_loader, criterion, DEVICE)

all_results['ACRNN'] = {
    'wa': wa_acrnn, 'ua': ua_acrnn,
    'preds': preds_acrnn, 'labels': labels_acrnn, 'history': hist_acrnn}
print(f"\nACRNN  |  Test WA: {wa_acrnn:.2%}  Test UA: {ua_acrnn:.2%}")


════════════════════════════════════════════════════════════════════
  ACRNN   |   lr=0.0003  wd=0.0003  patience=20
────────────────────────────────────────────────────────────────────
  Ep |   TrLoss |   TrWA |   VaLoss |   VaWA |   VaUA
────────────────────────────────────────────────────────────────────
   1 |   1.2078 | 24.55% |   1.1313 | 30.85% | 30.89%  ✓
   2 |   1.1197 | 30.56% |   1.0745 | 33.74% | 33.77%  ✓
   3 |   1.0591 | 34.76% |   0.9826 | 39.37% | 39.36%  ✓
   4 |   1.0042 | 39.17% |   0.9774 | 41.78% | 41.78%  ✓
   5 |   0.9338 | 44.64% |   0.8560 | 48.22% | 48.20%  ✓
   6 |   0.8966 | 47.02% |   0.8482 | 47.69% | 47.65%
   7 |   0.8595 | 49.33% |   0.7950 | 50.77% | 50.77%  ✓
   8 |   0.8228 | 52.90% |   0.7234 | 56.41% | 56.39%  ✓
   9 |   0.7981 | 53.95% |   0.8095 | 54.86% | 54.85%
  10 |   0.7534 | 56.75% |   0.7300 | 55.06% | 55.03%
  11 |   0.7030 | 59.74% |   0.8881 | 52.05% | 52.00%
  12 |   0.6745 | 61.26% |   0.6775 | 59.69% | 59.66%  ✓
  13 |   0.6442 | 

#### Model 4: AST-lite

##### Lightweight Audio Spectrogram Transformer.
###### `Reference: Gong et al., "AST: Audio Spectrogram Transformer", Interspeech 2021.`

- WHY a transformer for SER?
    - CNNs and RNNs have LOCAL receptive fields — a CNN kernel sees a small
    neighbourhood; an RNN accumulates context step-by-step. Both can miss
    long-range correlations (e.g., the pitch at second 0.5 correlating with
    the energy burst at second 3.0 in an angry sentence).

    - A Transformer's self-attention is GLOBAL: every patch attends to every
    other patch directly, with no path length constraint. This lets the model
    learn that low-frequency pitch patches and high-frequency breathiness
    patches co-occur in specific emotional patterns.

    - Patch size (8 mel × 25 frames):
      - 64/8 = 8 freq patches, 125/25 = 5 time patches → 40 tokens total
      - Each token covers ~160ms of audio and 8 mel bands — enough context
        to represent a vowel or consonant segment

In [ ]:
# Check the actual shape coming out of the dataloader
xb, yb = next(iter(train_loader))
print(f"Actual batch shape : {tuple(xb.shape)}")
print(f"N_MELS={N_MELS}, N_FRAMES (from data)={xb.shape[-1]}")

# Update N_FRAMES to match reality
N_FRAMES = xb.shape[-1]   # 126, not 125
print(f"\nN_FRAMES corrected to {N_FRAMES}")
print(f"Divisibility check:")
for pw in [6, 7, 9, 14, 18, 21, 42, 63]:
    if N_FRAMES % pw == 0:
        print(f"  patch_w={pw:>3}  →  npw={N_FRAMES//pw}  patches={N_MELS//8 * N_FRAMES//pw}")

: 

In [14]:
class PatchEmbedding(nn.Module):
    """Cut the spectrogram into non-overlapping patches and project each to d_model."""
    def __init__(self, patch_h=8, patch_w=25, d_model=128, in_channels=1):
        super().__init__()
        patch_dim = in_channels * patch_h * patch_w     # 1*8*25 = 200
        self.patch_h = patch_h
        self.patch_w = patch_w
        # Pre-LN → linear → post-LN (stabilises early training)
        self.proj = nn.Sequential(
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, d_model),
            nn.LayerNorm(d_model)
        )

    def forward(self, x):                           # (B, 1, 64, 125)
        B, C, H, W  = x.shape
        ph, pw      = self.patch_h, self.patch_w
        nph, npw    = H // ph, W // pw              # 8, 5

        # Reshape into patches without using unfold (cleaner gradient flow)
        x = x.reshape(B, C, nph, ph, npw, pw)       # (B,1,8,8,5,25)
        x = x.permute(0, 2, 4, 1, 3, 5).contiguous()# (B,8,5,1,8,25)
        x = x.reshape(B, nph * npw, C * ph * pw)     # (B, 40, 200)
        return self.proj(x)                          # (B, 40, 128)


class ASTLite(nn.Module):

    def __init__(self, num_classes=6, d_model=128, n_heads=4,
                 n_layers=4, d_ff=256, dropout=0.2,
                 patch_h=8, patch_w=25):
        super().__init__()

        n_patches = (N_MELS // patch_h) * (N_FRAMES // patch_w)   # 40

        self.patch_embed = PatchEmbedding(patch_h, patch_w, d_model)

        # CLS token: a learnable vector prepended to the patch sequence.
        # After transformer processing, it aggregates global information
        # and feeds into the classifier.
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        # Learnable positional encoding — each of the 41 positions gets
        # a unique learnable bias added to its embedding
        self.pos_embed  = nn.Parameter(torch.randn(1, n_patches + 1, d_model) * 0.02)

        self.drop = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, activation='gelu',
            batch_first=True, norm_first=True   # Pre-LN: more stable than Post-LN
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):                               # (B, 1, 64, 125)
        B       = x.size(0)
        tokens  = self.patch_embed(x)                   # (B, 40, 128)

        cls     = self.cls_token.expand(B, -1, -1)      # (B, 1,  128)
        tokens  = torch.cat([cls, tokens], dim=1)       # (B, 41, 128)
        tokens  = self.drop(tokens + self.pos_embed)    # add positional encoding

        tokens  = self.transformer(tokens)              # (B, 41, 128)
        tokens  = self.norm(tokens)

        cls_out = tokens[:, 0]                          # (B, 128)  — CLS token only
        return self.head(cls_out)                       # (B, 6)

_m = ASTLite(NUM_CLASSES)
_o = _m(torch.randn(4, 1, N_MELS, N_FRAMES))
print(f"AST-lite")
print(f"  Patches: {N_MELS//8}×{N_FRAMES//25} = {(N_MELS//8)*(N_FRAMES//25)} patches + 1 CLS = 41 tokens")
print(f"  Params : {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"  Output : {tuple(_o.shape)}")
del _m, _o

AST-lite
  Patches: 8×5 = 40 patches + 1 CLS = 41 tokens
  Params : 570,582
  Output : (4, 6)


/tmp/ipykernel_18552/3085449923.py:54: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


In [15]:
# Transformers need lower LR, more warmup, and less dropout than CNNs.
# On ~10k samples they can underfit — monitor val UA carefully.
cfg_ast = {
    'lr': 1e-4, 'wd': 1e-4, 'epochs': EPOCHS, 'warmup': 12, 'patience': 25}

model_ast, _, hist_ast = train_model(
    ASTLite(NUM_CLASSES, dropout=0.2), 'ASTLite', cfg_ast)

_, wa_ast, ua_ast, preds_ast, labels_ast = evaluate(
    model_ast, test_loader, criterion, DEVICE)

all_results['ASTLite'] = {
    'wa': wa_ast, 'ua': ua_ast,
    'preds': preds_ast, 'labels': labels_ast, 'history': hist_ast}
print(f"\nAST-lite  |  Test WA: {wa_ast:.2%}  Test UA: {ua_ast:.2%}")


════════════════════════════════════════════════════════════════════
  ASTLite   |   lr=0.0001  wd=0.0001  patience=25
────────────────────────────────────────────────────────────────────
  Ep |   TrLoss |   TrWA |   VaLoss |   VaWA |   VaUA
────────────────────────────────────────────────────────────────────


/tmp/ipykernel_18552/3085449923.py:54: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)


RuntimeError: shape '[32, 1, 8, 8, 5, 25]' is invalid for input of size 258048

#### Compare all 4 Models

In [ ]:
target_names = [INT_TO_LABEL[i] for i in range(NUM_CLASSES)]

# ── Summary table ──────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"  {'Model':<22}  {'Test WA':>8}  {'Test UA':>8}")
print(f"{'─'*60}")
for name, r in sorted(all_results.items(), key=lambda x: -x[1]['ua']):
    print(f"  {name:<22}  {r['wa']:>8.2%}  {r['ua']:>8.2%}")
print(f"{'='*60}")

# ── Training curves ────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Training history — Val UA vs Train WA", fontsize=13, fontweight='bold')
colors = {'ResNet18':'#7F77DD', 'CNN_BiLSTM_Attn':'#1D9E75',
          'ACRNN':'#D85A30', 'ASTLite':'#BA7517'}

for ax, (name, r) in zip(axes.flatten(), all_results.items()):
    h  = r['history']
    ep = range(1, len(h['tl']) + 1)
    ax.plot(ep, h['vu'], color=colors[name], lw=2,   label='Val UA')
    ax.plot(ep, h['va'], color=colors[name], lw=1.5, ls='--', label='Val WA')
    ax.plot(ep, h['ta'], color='#888780',    lw=1,   ls=':',  label='Train WA')
    ax.set_title(f"{name}  (UA={r['ua']:.2%})")
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9); ax.grid(alpha=0.3); ax.set_ylim(0.2, 1.0)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_curves_4models.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Confusion matrices ─────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Confusion matrices (normalised)", fontsize=13, fontweight='bold')

for ax, (name, r) in zip(axes.flatten(), all_results.items()):
    cm = confusion_matrix(r['labels'], r['preds'], normalize='true')
    sns.heatmap(cm, annot=True, fmt='.2f', cmap='Purples', ax=ax,
                xticklabels=target_names, yticklabels=target_names,
                linewidths=0.5, linecolor='white', vmin=0, vmax=1)
    ax.set_title(f"{name}  (UA={r['ua']:.2%})")
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrices_4models.png', dpi=150, bbox_inches='tight')
plt.show()

## Novel Model: MRTNet (Multi-Rate Temporal Network)

### Multi-Rate Temporal Network for Speech Emotion Recognition

---

#### Gap in Literature

Every model trained above — **ResNet, CNN-BiLSTM, ACRNN, AST** — applies the **same computational treatment to the entire mel-spectrogram**.  
They use **uniform spatial kernels across all 64 mel bands**.

However, **emotion is not uniformly distributed across frequency bands**.  
Each band encodes a **different aspect of emotion at a different temporal rate**:

##### Low Bands (mel 0–21)
- **Fundamental frequency (F0)** and pitch  
- **Prosodic contours** (e.g., rising or falling intonation)  
- These are **slow signals**, changing over **100–500 ms**  
- Require a **wide temporal receptive field**

##### Mid Bands (mel 22–43)
- **Formants (F1, F2)** — vocal tract resonances  
- **Vowel articulation** and **vocal effort**  
- **Medium temporal rate**, changing over **30–100 ms**

##### High Bands (mel 44–63)
- **Voice quality features** such as:
  - Breathiness
  - Creakiness
  - Fricative energy  
- **Fast transients (< 30 ms)**  
- Require a **narrow temporal receptive field**

---

#### Limitation of Existing Speech Emotion Recognition (SER) Models

No existing SER paper:

1. **Explicitly separates the spectrogram by frequency band**
2. **Uses rate-matched temporal kernels per band**
3. **Models cross-band temporal interactions with attention**

---

#### Proposed Architecture

    ┌─────────────┐   ┌─────────────┐   ┌─────────────┐
    │   Low Band  │   │   Mid Band  │   │  High Band  │
    │  (mel 0–21) │   │ (mel 22–43) │   │ (mel 44–63) │
    │  kernel = 7 │   │  kernel = 5 │   │  kernel = 3 │
    │    BiLSTM   │   │    BiLSTM   │   │    BiLSTM   │
    └──────┬──────┘   └──────┬──────┘   └──────┬──────┘
           └─────────────────┼─────────────────┘
                  Cross-Band Attention Transformer
             (bands attend to each other at each time step)
                             │
                    Gated Temporal Fusion
                             │
                          Classifier


---

#### Novel Contributions

- **Rate-matched temporal kernels**  
  Each frequency branch uses a **temporal kernel suited to its signal rate**.

- **Cross-band attention mechanism**  
  Captures **inter-band emotional dynamics**, e.g.:

  > High-frequency breathiness (fear) co-occurring with a low-frequency pitch drop.

- **Gated temporal fusion**  
  Allows the model to **learn per-sample band importance dynamically**.


In [ ]:
class MRTNet(nn.Module):

    def __init__(self, num_classes=6, d_model=128, dropout=0.35):
        super().__init__()

        # Frequency band boundaries
        self.bands = {'low': (0, 22), 'mid': (22, 44), 'high': (44, 64)}
        lstm_hidden = d_model // 2    # bidirectional → hidden*2 = d_model

        def make_branch(temporal_kernel):
            """
            2-block CNN + AdaptivePool — collapses freq to 1 then feeds BiLSTM.
            temporal_kernel controls the temporal receptive field per band.
            """
            pad = temporal_kernel // 2
            return nn.Sequential(
                # Block 1: local pattern detection
                nn.Conv2d(1, 32, kernel_size=(3, temporal_kernel),
                          padding=(1, pad), bias=False),
                nn.BatchNorm2d(32), nn.GELU(),
                nn.MaxPool2d(kernel_size=(2, 1)),      # halve freq, keep time

                # Block 2: deeper features
                nn.Conv2d(32, 64, kernel_size=(3, temporal_kernel),
                          padding=(1, pad), bias=False),
                nn.BatchNorm2d(64), nn.GELU(),

                # Collapse remaining freq to 1
                nn.AdaptiveAvgPool2d((1, N_FRAMES))    # (B, 64, 1, 125)
            )

        # Three branches — SAME structure, DIFFERENT temporal kernel
        self.branch_low  = make_branch(temporal_kernel=7)   # wide — slow prosody
        self.branch_mid  = make_branch(temporal_kernel=5)   # medium — formants
        self.branch_high = make_branch(temporal_kernel=3)   # narrow — voice quality

        # Independent BiLSTM per branch
        def make_bilstm():
            return nn.LSTM(64, lstm_hidden, num_layers=2,
                           batch_first=True, bidirectional=True,
                           dropout=dropout)

        self.lstm_low  = make_bilstm()   # output: (B, 125, d_model)
        self.lstm_mid  = make_bilstm()
        self.lstm_high = make_bilstm()

        # Cross-Band Attention Transformer
        # At each time step: 3 band tokens → attend across bands
        # Learns "how does pitch pattern here relate to voice quality here?"
        cross_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=4, dim_feedforward=d_model * 2,
            dropout=dropout, batch_first=True,
            norm_first=True, activation='gelu')
        self.cross_attn = nn.TransformerEncoder(cross_layer, num_layers=2)

        # Temporal attention: select key time steps per band after cross-attention
        self.time_score = nn.Linear(d_model, 1)

        # Gated fusion: learns which bands matter for THIS sample
        self.gate          = nn.Sequential(
            nn.Linear(d_model * 3, d_model * 3), nn.Sigmoid())
        self.fusion_proj   = nn.Linear(d_model * 3, d_model)
        self.norm          = nn.LayerNorm(d_model)
        self.drop          = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def _branch_forward(self, x_full, band, cnn, lstm):
        """Slice frequency band → CNN → BiLSTM → (B, T, d_model)."""
        s, e = band
        x        = x_full[:, :, s:e, :]        # (B, 1, band_size, 125)
        x        = cnn(x)                       # (B, 64, 1, 125)
        x        = x.squeeze(2).permute(0,2,1)  # (B, 125, 64)
        x, _     = lstm(x)                      # (B, 125, d_model)
        return x

    def forward(self, x):       # x: (B, 1, 64, 125)

        # 1. Process each frequency band independently
        low  = self._branch_forward(x, self.bands['low'],  self.branch_low,  self.lstm_low)
        mid  = self._branch_forward(x, self.bands['mid'],  self.branch_mid,  self.lstm_mid)
        high = self._branch_forward(x, self.bands['high'], self.branch_high, self.lstm_high)
        # each: (B, 125, 128)

        # 2. Cross-Band Attention at every time step
        # Stack the 3 bands as tokens → transformer sees all 3 bands per step
        B, T, D = low.shape
        bands   = torch.stack([low, mid, high], dim=2)  # (B, T, 3, D)
        bands   = bands.reshape(B * T, 3, D)            # (B*T, 3, D)
        bands   = self.cross_attn(bands)                # (B*T, 3, D)
        bands   = bands.reshape(B, T, 3, D)             # (B, T, 3, D)

        # 3. Temporal attention pool each band
        def t_pool(feat):   # (B, T, D) → (B, D)
            w = torch.softmax(self.time_score(feat), dim=1)  # (B, T, 1)
            return (w * feat).sum(dim=1)                     # (B, D)

        v_low  = t_pool(bands[:, :, 0, :])   # (B, D)
        v_mid  = t_pool(bands[:, :, 1, :])
        v_high = t_pool(bands[:, :, 2, :])

        # 4. Gated fusion: learn which bands matter for this input
        cat    = torch.cat([v_low, v_mid, v_high], dim=-1)   # (B, 3D)
        fused  = self.fusion_proj(self.gate(cat) * cat)       # (B, D)
        fused  = self.norm(self.drop(fused))

        return self.head(fused)

_m = MRTNet(NUM_CLASSES)
_o = _m(torch.randn(4, 1, N_MELS, N_FRAMES))
print(f"MRTNet (Multi-Rate Temporal Network)")
print(f"  Params : {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
print(f"  Output : {tuple(_o.shape)}")
print(f"\n  Branches:")
print(f"    Low  (mel 0-21)  → temporal_kernel=7  (slow prosody)")
print(f"    Mid  (mel 22-43) → temporal_kernel=5  (formants)")
print(f"    High (mel 44-63) → temporal_kernel=3  (voice quality)")
del _m, _o

In [ ]:
cfg_mrt = {
    'lr': 3e-4, 'wd': 3e-4, 'epochs': EPOCHS, 'warmup': 10, 'patience': 22}

model_mrt, _, hist_mrt = train_model(
    MRTNet(NUM_CLASSES, dropout=0.35), 'MRTNet', cfg_mrt)

_, wa_mrt, ua_mrt, preds_mrt, labels_mrt = evaluate(
    model_mrt, test_loader, criterion, DEVICE)

all_results['MRTNet'] = {
    'wa': wa_mrt, 'ua': ua_mrt,
    'preds': preds_mrt, 'labels': labels_mrt, 'history': hist_mrt}
print(f"\nMRTNet  |  Test WA: {wa_mrt:.2%}  Test UA: {ua_mrt:.2%}")

#### Final Comparison

In [ ]:
target_names = [INT_TO_LABEL[i] for i in range(NUM_CLASSES)]

# ── Summary table ──────────────────────────────────────────────
print(f"\n{'='*65}")
print(f"  {'Model':<22}  {'Test WA':>8}  {'Test UA':>8}  {'Design'}") 
print(f"{'─'*65}")
designs = {
    'ResNet18':        'Pure 2D CNN',
    'CNN_BiLSTM_Attn': 'CNN + temporal RNN + attention',
    'ACRNN':           'Freq-attn + RNN + time-attn',
    'ASTLite':         'Patch transformer',
    'MRTNet':          '★ Multi-rate + cross-band attn (novel)',
}
for name, r in sorted(all_results.items(), key=lambda x: -x[1]['ua']):
    print(f"  {name:<22}  {r['wa']:>8.2%}  {r['ua']:>8.2%}  {designs[name]}")
print(f"{'='*65}")

# ── Bar chart ──────────────────────────────────────────────────
names = list(all_results.keys())
was   = [all_results[n]['wa'] * 100 for n in names]
uas   = [all_results[n]['ua'] * 100 for n in names]
bar_colors = ['#7F77DD','#1D9E75','#D85A30','#BA7517','#E8593C']

x = np.arange(len(names)); w = 0.35
fig, ax = plt.subplots(figsize=(13, 5))
fig.suptitle("Final model comparison — Test set", fontsize=13, fontweight='bold')
bars_wa = ax.bar(x - w/2, was, w, label='WA (%)', alpha=0.85, color=bar_colors)
bars_ua = ax.bar(x + w/2, uas, w, label='UA (%)', alpha=0.85,
                 color=bar_colors, edgecolor='white', linewidth=1.5)

for bar in list(bars_wa) + list(bars_ua):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9)

ax.axvline(x=3.5, color='#888780', ls='--', lw=0.8, label='Novel model boundary')
ax.set_xticks(x); ax.set_xticklabels(names, rotation=15, ha='right')
ax.set_ylabel('Accuracy (%)'); ax.set_ylim(40, 100)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'grand_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Full report for best model ─────────────────────────────────
best_name = max(all_results, key=lambda k: all_results[k]['ua'])
r         = all_results[best_name]
print(f"\nDetailed report — {best_name}:")
print(classification_report(r['labels'], r['preds'],
                             target_names=target_names, digits=4))

# ── Save everything ────────────────────────────────────────────
model_objs = {
    'ResNet18': model_r18, 'CNN_BiLSTM_Attn': model_clba,
    'ACRNN': model_acrnn,  'ASTLite': model_ast, 'MRTNet': model_mrt}

for name, model in model_objs.items():
    torch.save(model.state_dict(), OUTPUT_DIR / f'{name}_final.pt')

summary = {name: {'test_wa': round(r['wa'],4), 'test_ua': round(r['ua'],4)}
           for name, r in all_results.items()}
summary['best_model'] = best_name
with open(OUTPUT_DIR / 'final_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\nAll models saved to {OUTPUT_DIR.resolve()}")